# Exp 2 — BoW + W&B Sweep

In [10]:
!pip install datasets wandb scikit-learn sentence-transformers gensim -q

In [11]:
import torch, torch.nn as nn, torch.optim as optim, torch.backends.cudnn as cudnn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score
from datasets import load_dataset
import numpy as np, copy, wandb
SEED=42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cudnn.benchmark=False; cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{device}')

Device:cuda


In [12]:
data=load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
def remove_empty(row):
    return all(row[f] not in [None,''] for f in ['id','text','label','sentiment'])
train_data=data['train'].filter(remove_empty)
dev_data=data['validation'].filter(remove_empty)
test_data=data['test'].filter(remove_empty)
output_size=len(set(train_data['label']))
train_labels=train_data['label']
test_labels_list=test_data['label']
print(f'Train:{len(train_data)}, Dev:{len(dev_data)}, Test:{len(test_data)}')

Train:31232, Dev:5205, Test:5205


In [13]:
vectorizer=CountVectorizer()
vectorizer.fit(train_data['text'])
train_v=vectorizer.transform(train_data['text'])
dev_v=vectorizer.transform(dev_data['text'])
test_v=vectorizer.transform(test_data['text'])
input_size=train_v.shape[1]
print(f'BoW vocab:{input_size}')
train_t=torch.FloatTensor(train_v.toarray()).to(device)
dev_t=torch.FloatTensor(dev_v.toarray()).to(device)
test_t=torch.FloatTensor(test_v.toarray()).to(device)
dev_labels_t=torch.tensor(dev_data['label'],dtype=torch.long).to(device)

BoW vocab:29053
[Exp2-BoW] Dev:0.6740|Test:66.57%


In [14]:
class MLP(nn.Module):
    def __init__(self, i, h, o, d=0.0):
        super().__init__()
        self.fc1=nn.Linear(i,h)
        self.fc2=nn.Linear(h,h//2)
        self.fc3=nn.Linear(h//2,o)
        self.activation=nn.GELU()
        self.output_act=nn.Softmax(dim=1)
        self.dropout=nn.Dropout(p=d)
    def forward(self,x):
        x=self.dropout(self.activation(self.fc1(x)))
        x=self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

In [15]:
def make_sweep_fn(train_t,dev_t,dev_l,test_t,inp,lbl):
    def train_fn():
        with wandb.init() as run:
            cfg=run.config
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
            model=MLP(inp,cfg.hidden_size,output_size,cfg.dropout).to(device)
            opt=optim.Adam(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
            lfn=nn.CrossEntropyLoss()
            best_dev,best_state=0,None
            for epoch in range(cfg.num_epochs):
                model.train()
                eloss=0
                for i in range(0,len(train_t),cfg.batch_size):
                    bd=train_t[i:i+cfg.batch_size]
                    bl=torch.tensor(train_labels[i:i+cfg.batch_size],device=device)
                    out=model(bd)
                    loss=lfn(out,bl)
                    opt.zero_grad(); loss.backward(); opt.step()
                    eloss+=loss.item()
                model.eval()
                with torch.no_grad():
                    da=(torch.argmax(model(dev_t),dim=1)==dev_l).float().mean().item()
                if da>best_dev:
                    best_dev=da
                    best_state=copy.deepcopy(model.state_dict())
                wandb.log({'epoch':epoch+1,'dev_accuracy':da,'best_dev_accuracy':best_dev,'train_loss':eloss/len(train_t)})
            model.load_state_dict(best_state)
            model.eval()
            with torch.no_grad():
                tp=torch.argmax(model(test_t),dim=1)
                ta=accuracy_score(test_labels_list,tp.cpu().tolist())
            wandb.log({'test_accuracy':ta})
            print(f'[{lbl}] Dev:{best_dev:.4f}|Test:{ta*100:.2f}%')
    return train_fn

SWEEP_CFG={'method':'bayes','metric':{'name':'best_dev_accuracy','goal':'maximize'},
'parameters':{'learning_rate':{'distribution':'log_uniform_values','min':1e-5,'max':1e-2},
'hidden_size':{'values':[256,512,1000,2000]},'dropout':{'values':[0.0,0.1,0.2,0.3,0.5]},
'weight_decay':{'values':[0.0,1e-5,1e-4,1e-3]},'num_epochs':{'values':[20,30,50]},
'batch_size':{'values':[64,128,256]}}}

In [16]:
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [17]:
sweep_id=wandb.sweep({**SWEEP_CFG,'name':'exp2-bow'},project='nlp-hw1')
print(f'Sweep ID:{sweep_id}')
train_fn=make_sweep_fn(train_t,dev_t,dev_labels_t,test_t,input_size,'Exp2-BoW')
wandb.agent(sweep_id,function=train_fn,count=20)

best_dev_accuracy,▁▃▃▃▃▃▃▃▃▃▃▃▅▆▆▆▆▆▇▇▇█████████
dev_accuracy,▃▅▁▁▃▃▂▄▃▃▄▄▆▆▅▅▄▆▇▇▆█▅▆▄▅▆▆█▆
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
test_accuracy,▁
train_loss,█▆▆▅▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
best_dev_accuracy,0.67397
dev_accuracy,0.66475
epoch,30
test_accuracy,0.66571
train_loss,0.00537


Create sweep with ID: gzg4mb69
Sweep URL: https://wandb.ai/imeanseo_/nlp-hw1/sweeps/gzg4mb69
Sweep ID:gzg4mb69


wandb: Agent Starting Run: 4setnw88 with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0.1
wandb: 	hidden_size: 512
wandb: 	learning_rate: 1.6531967983458977e-05
wandb: 	num_epochs: 30
wandb: 	weight_decay: 1e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp2-BoW] Dev:0.6872|Test:68.05%


best_dev_accuracy,▁▁▃▄▅▆▆▆▆▇▇▇▇█████████████████
dev_accuracy,▁▁▃▄▅▆▆▆▆▇▇▇▇█████████████████
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
test_accuracy,▁
train_loss,███▇▇▆▆▆▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
best_dev_accuracy,0.68722
dev_accuracy,0.68569
epoch,30
test_accuracy,0.6805
train_loss,0.00292


...

[Exp2-BoW] Dev:0.6924|Test:67.88%


best_dev_accuracy,▁▅▆▇████████████████████████████████████
dev_accuracy,▁▅▆▇████████████████████████████████████
epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
test_accuracy,▁
train_loss,█▇▆▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.69241
dev_accuracy,0.68242
epoch,50
test_accuracy,0.67877
train_loss,0.00256


wandb: Agent Starting Run: wioze87h with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0
wandb: 	hidden_size: 1000
wandb: 	learning_rate: 1.1121385418843466e-05
wandb: 	num_epochs: 30
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp2-BoW] Dev:0.6926|Test:68.17%


best_dev_accuracy,▁▂▄▄▅▅▆▆▆▇▇▇▇▇████████████████
dev_accuracy,▁▂▄▄▅▅▆▆▆▇▇▇▇▇████████████████
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
test_accuracy,▁
train_loss,████▇▆▆▆▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
best_dev_accuracy,0.6926
dev_accuracy,0.6926
epoch,30
test_accuracy,0.68165
train_loss,0.00296


In [19]:
USERNAME='imeanseo_'
api=wandb.Api()
sw=api.sweep(f'{USERNAME}/nlp-hw1/{sweep_id}')
best=sw.best_run()
print('\n'+'='*60)
print('🏆 Best Run Config:')
for k,v in dict(best.config).items():
    print(f'  {k:<20}: {v}')
print(f"\nBest Dev:{best.summary['best_dev_accuracy']:.4f}")
print(f"Test:{best.summary['test_accuracy']*100:.2f}%")
print('='*60)

wandb: Sorting runs by -summary_metrics.best_dev_accuracy



🏆 Best Run Config:
  dropout             : 0.1
  batch_size          : 256
  num_epochs          : 50
  hidden_size         : 512
  weight_decay        : 0.0001
  learning_rate       : 6.349241367062284e-05

Best Dev:0.6945
Test:68.26%


## Best Config로 재학습 + 저장

In [20]:
# ⚠️ 위 출력 값으로 수정
BEST_H,BEST_LR,BEST_D,BEST_WD,BEST_EP,BEST_BS=1000,0.0003,0.2,1e-4,30,128
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
final=MLP(input_size,BEST_H,output_size,BEST_D).to(device)
opt=optim.Adam(final.parameters(),lr=BEST_LR,weight_decay=BEST_WD)
lfn=nn.CrossEntropyLoss()
best_dev,best_state=0,None
for epoch in range(BEST_EP):
    final.train()
    for i in range(0,len(train_t),BEST_BS):
        bd=train_t[i:i+BEST_BS]
        bl=torch.tensor(train_labels[i:i+BEST_BS],device=device)
        loss=lfn(final(bd),bl)
        opt.zero_grad(); loss.backward(); opt.step()
    final.eval()
    with torch.no_grad():
        da=(torch.argmax(final(dev_t),dim=1)==dev_labels_t).float().mean().item()
    if da>best_dev:
        best_dev,best_state=da,copy.deepcopy(final.state_dict())
    print(f'Epoch {epoch+1}/{BEST_EP}|Dev:{da:.4f}')
final.load_state_dict(best_state)
torch.save(best_state,'best_model_exp2.pt')
with torch.no_grad():
    test_acc=accuracy_score(test_labels_list,torch.argmax(final(test_t),dim=1).cpu().tolist())
print(f'\n✅ 저장:best_model_exp2.pt|Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%')

Epoch 1/30|Dev:0.6792
Epoch 2/30|Dev:0.6872
Epoch 3/30|Dev:0.6801
Epoch 4/30|Dev:0.6747
Epoch 5/30|Dev:0.6688
Epoch 6/30|Dev:0.6797
Epoch 7/30|Dev:0.6768
Epoch 8/30|Dev:0.6734
Epoch 9/30|Dev:0.6632
Epoch 10/30|Dev:0.6701
Epoch 11/30|Dev:0.6703
Epoch 12/30|Dev:0.6728
Epoch 13/30|Dev:0.6717
Epoch 14/30|Dev:0.6619
Epoch 15/30|Dev:0.6661
Epoch 16/30|Dev:0.6669
Epoch 17/30|Dev:0.6682
Epoch 18/30|Dev:0.6592
Epoch 19/30|Dev:0.6605
Epoch 20/30|Dev:0.6647
Epoch 21/30|Dev:0.6630
Epoch 22/30|Dev:0.6697
Epoch 23/30|Dev:0.6682
Epoch 24/30|Dev:0.6709
Epoch 25/30|Dev:0.6680
Epoch 26/30|Dev:0.6615
Epoch 27/30|Dev:0.6651
Epoch 28/30|Dev:0.6680
Epoch 29/30|Dev:0.6674
Epoch 30/30|Dev:0.6655

✅ 저장:best_model_exp2.pt|Dev:0.6872|Test:67.86%


In [22]:
from google.colab import files
files.download('best_model_exp2.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>